# Register S3

In [2]:
import boto3
import sagemaker

sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [3]:
ingest_create_athena_table_emotions_passed = False

In [4]:
%store -r ingest_create_athena_db_passed

In [5]:
try:
    ingest_create_athena_db_passed
except NameError:
    print("++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL PREVIOUS NOTEBOOKS.  You did not create the Athena Database.")
    print("++++++++++++++++++++++++++++++++++++++++++++++")

In [6]:
print(ingest_create_athena_db_passed)

True


In [7]:
if not ingest_create_athena_db_passed:
    print("++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN ALL PREVIOUS NOTEBOOKS.  You did not create the Athena Database.")
    print("++++++++++++++++++++++++++++++++++++++++++++++")
else:
    print("[OK]")

[OK]


In [8]:
%store -r s3_private_path_wav

In [9]:
try:
    s3_private_path_wav
except NameError:
    print("*****************************************************************************")
    print("[ERROR] PLEASE RE-RUN THE PREVIOUS COPY TSV TO S3 NOTEBOOK ******************")
    print("[ERROR] THIS NOTEBOOK WILL NOT RUN PROPERLY. ********************************")
    print("*****************************************************************************")

In [10]:
print(s3_private_path_wav)

s3://sagemaker-us-east-1-218117716191/audio/Datasets/


## Import PyAthena

In [11]:
from pyathena import connect

## Create Athena Table from Local Files

In [12]:
glue = boto3.client('glue', region_name='us-east-1')
s3_path = "s3://sagemaker-us-east-1-218117716191/audio/Datasets/"

## Create Crawler in AWS Glue console to learn dB Schema

Console Method
1. AWS Console > Glue > Crawlers > Add crawler.

2. Name: audio_datasets_crawler.

3. Data source: S3 > s3://sagemaker-us-east-1-218117716191/audio/Datasets/.

4. IAM role: Select/create AWSGlueServiceRole (auto-permissions).

5. Database: audio_emotions.

6. Run crawler—tables appear in Athena in a few minutes.

In [17]:
#Check status of crawler
glue_client = boto3.client('glue', region_name='us-east-1')

crawler = glue_client.get_crawler(Name='audio_datasets_crawler')
print("Crawler State:", crawler['Crawler']['State'])

# Fixed metrics call
metrics = glue_client.get_crawler_metrics(CrawlerNameList=['audio_datasets_crawler'])  # NameList!
runs = metrics['CrawlerMetricsList'][0]['CrawlerRunCount']
print("Runs:", runs['TotalRuns'], "Successful:", runs['SuccessfulRuns'])

# Poll
import time
while crawler['Crawler']['State'] == 'RUNNING':
    time.sleep(30)
    crawler = glue_client.get_crawler(Name='audio_datasets_crawler')
print("Ready—check tables!")

Crawler State: READY


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:9                                                                                    │
│                                                                                                  │
│    6                                                                                             │
│    7 # Fixed metrics call                                                                        │
│    8 metrics = glue_client.get_crawler_metrics(CrawlerNameList=['audio_datasets_crawler'])  #    │
│ ❱  9 runs = metrics['CrawlerMetricsList'][0]['CrawlerRunCount']                                  │
│   10 print("Runs:", runs['TotalRuns'], "Successful:", runs['SuccessfulRuns'])                    │
│   11                                                                                             │
│   12 # Poll                                                                                      │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
KeyError: 'CrawlerRunCount'

## Verify Table Has Been Created

In [18]:
# List tables in database
tables = glue_client.get_tables(DatabaseName='audio_emotions')
print("Tables Found:", [t['Name'] for t in tables['TableList']])

Tables Found: ['cleaned_age_csv', 'cleaned_emotion_csv', 'cleaned_gender_csv', 'emotions']


In [20]:
# Query via Athena
import pandas as pd
conn = connect(s3_staging_dir='s3://sagemaker-us-east-1-218117716191/athena/staging/', region_name='us-east-1')
df = pd.read_sql("SHOW TABLES IN audio_emotions", conn)
df  # Displays inferred CSV tables

/tmp/ipykernel_1463/3509945352.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SHOW TABLES IN audio_emotions", conn)


,tab_name
0,cleaned_age_csv
1,cleaned_emotion_csv
2,cleaned_gender_csv
3,emotions


## Run A Sample Query

In [24]:
df_all = pd.read_sql("SHOW DATABASES", conn)
df_all

/tmp/ipykernel_1463/1031642697.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_all = pd.read_sql("SHOW DATABASES", conn)


,database_name
0,am2aws
1,audio_emotions
2,default
3,dsoaws
4,hm2aws


In [29]:
# Confirm tables in audio_emotions
df_tables = pd.read_sql("SHOW TABLES IN audio_emotions", conn)
print(df_tables)

# Sample rows from each
queries = [
    "SELECT * FROM audio_emotions.cleaned_age_csv LIMIT 5",
    "SELECT * FROM audio_emotions.cleaned_emotion_csv LIMIT 5",
    "SELECT * FROM audio_emotions.cleaned_gender_csv LIMIT 5"
]

for q in queries:
    print("\n", q)
    display(pd.read_sql(q, conn))

/tmp/ipykernel_1463/3102977200.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_tables = pd.read_sql("SHOW TABLES IN audio_emotions", conn)


              tab_name
0      cleaned_age_csv
1  cleaned_emotion_csv
2   cleaned_gender_csv
3             emotions

 SELECT * FROM audio_emotions.cleaned_age_csv LIMIT 5


/tmp/ipykernel_1463/3102977200.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql(q, conn))


,col0,col1,col2,col3,col4,col5,col6,col7,col8,col9,...,col13,col14,col15,col16,col17,col18,col19,col20,col21,col22



 SELECT * FROM audio_emotions.cleaned_emotion_csv LIMIT 5


/tmp/ipykernel_1463/3102977200.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql(q, conn))


,col0,col1,col2,col3,col4,col5,col6,col7,col8,col9,...,col14,col15,col16,col17,col18,col19,col20,col21,col22,col23



 SELECT * FROM audio_emotions.cleaned_gender_csv LIMIT 5


/tmp/ipykernel_1463/3102977200.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  display(pd.read_sql(q, conn))


,col0,col1,col2,col3,col4,col5,col6,col7,col8,col9,...,col12,col13,col14,col15,col16,col17,col18,col19,col20,col21


## Review the New Athena Table in Glue Catalog

In [31]:
from IPython.core.display import display, HTML

region = "us-east-1"
database = "audio_emotions"

glue_url = (
    f"https://{region}.console.aws.amazon.com/glue/home"
    f"?region={region}#catalog:tab=databases;database={database}"
)

display(
    HTML(
        f'<b>Review <a target="_blank" href="{glue_url}">AWS Glue Catalog (audio_emotions)</a></b>'
    )
)

/tmp/ipykernel_1463/2980283929.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


## Store Variables for the Next Notebooks

In [32]:
%store

Stored variables and their in-db values:
ingest_create_athena_db_passed             -> True
s3_file_path_wav                           -> 's3://sagemaker-us-east-1-218117716191/audio/Datas
s3_private_path_wav                        -> 's3://sagemaker-us-east-1-218117716191/audio/Datas
setup_dependencies_passed                  -> True
setup_s3_bucket_passed                     -> True


## Release Resources

In [33]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>

In [1]:
%%javascript

try {
    Jupyter.notebook.save_checkpoint();
    Jupyter.notebook.session.delete();
}
catch(err) {
    // NoOp
}

<IPython.core.display.Javascript object>